# Batch, Epoch, DataLoader
딥러닝에서 데이터를 어떻게 나누어 모델에 넣는지 살펴본다.

## 데이터를 나누어 넣는 이유
딥러닝에서는 전체 데이터를 한 번에 넣어 학습하는 경우보다, 여러 개씩 나누어 반복 학습하는 경우가 더 일반적이다.

1. 데이터가 많을수록 한 번에 다 올리면 메모리 부담이 크다.
2. 여러 번 나누어 업데이트하면 학습이 더 현실적이고 유연하다.
3. 딥러닝 프레임워크는 batch 단위 학습을 기본 흐름으로 가정한다.

## 용어 정리
1. sample
- 데이터 1개를 의미한다.
- 예: 학생 한 명의 정보, 이미지 한 장, 문장 하나
2. batch
- 한 번에 모델에 넣는 sample 묶음이다.
3. batch size
- 한 batch 안에 들어 있는 sample 개수이다.
4. iteration(step)
- batch 1개를 처리하는 한 번의 반복이다.
- 보통 batch 하나에 대해 예측을 하고, 손실을 계산하고, 파라미터를 업데이트하는 흐름을 1 iteration이라고 본다.
5. epoch
- 전체 학습 데이터를 처음부터 끝까지 한 번 모두 사용한 것이다.

## 간단한 toy 데이터로 테스트
batch 개념을 보기 쉽도록 작은 tensor 데이터를 직접 만든다.
- `X` : 입력 데이터
- `y` : 정답 라벨

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)   # 랜덤 시드 고정

# sample 10개, feature 2개
X = torch.arange(20).reshape(10, 2).float()

# 각 샘플에 대응하는 label 10개
y = torch.arange(10)

print(X, X.shape)
print(y, y.shape)

tensor([[ 0.,  1.],
        [ 2.,  3.],
        [ 4.,  5.],
        [ 6.,  7.],
        [ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.],
        [16., 17.],
        [18., 19.]]) torch.Size([10, 2])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) torch.Size([10])


## TensorDataset 만들기
입력과 정답을 각각 따로 들고 있기보다, sample 단위로 짝지어 dataset 형태로 묶는 경우가 많다.

In [2]:
dataset = TensorDataset(X, y)

print('dataset의 길이 : ', len(dataset))
print('첫 번째 sample', dataset[0])
print('두 번째 sample', dataset[1])

dataset의 길이 :  10
첫 번째 sample (tensor([0., 1.]), tensor(0))
두 번째 sample (tensor([2., 3.]), tensor(1))


## DataLoader 만들기
dataset을 batch 단위로 나누고, 반복문으로 꺼내기 쉽게 만들어준다.
- `batch_size` : 한 번에 꺼낼 sample 수
- `shuffle`: epoch가 시작 될 때 sample 순서를 섞을지 여부

In [3]:
loader = DataLoader(dataset, batch_size=4, shuffle=False)
loader

## DataLoader 순회하기

In [4]:
for batch_idx, (batch_X, batch_y) in enumerate(loader, start=1):
    print(f'batch {batch_idx}')
    print(f'batch_X shape : {batch_X.shape}')
    print(f'batch_y shape : {batch_y.shape}')
    print('batch_X')
    print(batch_X)
    print('batch_y')
    print(batch_y)
    print()

batch 1
batch_X shape : torch.Size([4, 2])
batch_y shape : torch.Size([4])
batch_X
tensor([[0., 1.],
        [2., 3.],
        [4., 5.],
        [6., 7.]])
batch_y
tensor([0, 1, 2, 3])

batch 2
batch_X shape : torch.Size([4, 2])
batch_y shape : torch.Size([4])
batch_X
tensor([[ 8.,  9.],
        [10., 11.],
        [12., 13.],
        [14., 15.]])
batch_y
tensor([4, 5, 6, 7])

batch 3
batch_X shape : torch.Size([2, 2])
batch_y shape : torch.Size([2])
batch_X
tensor([[16., 17.],
        [18., 19.]])
batch_y
tensor([8, 9])



In [5]:
loader_shuffle = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_idx, (_, batch_y) in enumerate(loader_shuffle, start=1):
    print(f'batch {batch_idx} labels: {batch_y.tolist()}')

batch 1 labels: [4, 3, 2, 6]
batch 2 labels: [7, 5, 0, 8]
batch 3 labels: [1, 9]


## epoch 단위로 DataLoader 반복하기

In [7]:
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

for epoch in range(2):
    print(f'epoch {epoch + 1} 시작')

    for batch_idx, (batch_X, batch_y) in enumerate(train_loader, start=1):
        print(f'batch {batch_idx} -> X shape: {batch_X.shape}, y shape : {batch_y.shape}')

    print()

epoch 1 시작
batch 1 -> X shape: torch.Size([4, 2]), y shape : torch.Size([4])
batch 2 -> X shape: torch.Size([4, 2]), y shape : torch.Size([4])
batch 3 -> X shape: torch.Size([2, 2]), y shape : torch.Size([2])

epoch 2 시작
batch 1 -> X shape: torch.Size([4, 2]), y shape : torch.Size([4])
batch 2 -> X shape: torch.Size([4, 2]), y shape : torch.Size([4])
batch 3 -> X shape: torch.Size([2, 2]), y shape : torch.Size([2])



# train / valdation loader
실제 학습에서는 보통 학습 데이터와 검증 데이터를 나눠서 사용한다.

In [9]:
X_train, y_train = X[:8], y[:8]
X_val, y_val = X[8:], y[8:]

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

for batch_X, batch_y in train_loader:
    print('train batch ->', batch_X.shape, batch_y.shape)

for batch_X, batch_y in val_loader:
    print('validation batch ->', batch_X.shape, batch_y.shape)

train batch -> torch.Size([4, 2]) torch.Size([4])
train batch -> torch.Size([4, 2]) torch.Size([4])
validation batch -> torch.Size([2, 2]) torch.Size([2])
